In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import os
os.chdir("/content")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# !rm -rf "/content/vlm-safety-reasoning"

In [ ]:
import subprocess
import shutil

DRIVE_ROOT = "/content/drive/MyDrive/vlm-finetuning-project1"
REPO_DIR = "vlm-safety-reasoning"
ENV_PATH = f"{DRIVE_ROOT}/secrets/.env"

def load_secrets(env_path: str) -> dict:
    """Read a .env file and export its values into os.environ."""
    if not os.path.exists(env_path):
        raise FileNotFoundError(f"Secrets file not found at: {env_path}")

    secrets = {}
    with open(env_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            secrets[key] = value.strip(" \"'\r")
            os.environ[key] = secrets[key]
    return secrets


print(">>> Loading secrets...")
secrets = load_secrets(ENV_PATH)
required_keys = ["GIT_EMAIL", "GIT_NAME", "GITHUB_USERNAME", "GITHUB_TOKEN", "HF_TOKEN"]
missing = [k for k in required_keys if k not in secrets]
if missing:
    raise KeyError(f"Missing required secrets: {missing}")
print(">>> Secrets loaded successfully.")

print(">>> Configuring Git identity...")
subprocess.run(["git", "config", "--global", "user.email", secrets["GIT_EMAIL"]], check=True)
subprocess.run(["git", "config", "--global", "user.name", secrets["GIT_NAME"]], check=True)

AUTH_REPO_URL = (
    f"https://{secrets['GITHUB_USERNAME']}:{secrets['GITHUB_TOKEN']}"
    f"@github.com/epmresearch/vlm-safety-reasoning.git"
)

if os.path.exists(REPO_DIR):
    print(">>> Repo already present, pulling latest...")
    os.chdir(REPO_DIR)
    subprocess.run(["git", "remote", "set-url", "origin", AUTH_REPO_URL], check=True)
    subprocess.run(["git", "pull", "origin", "main"], check=True)
else:
    print(">>> Cloning repo...")
    subprocess.run(["git", "clone", AUTH_REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print(f">>> Working directory: {os.getcwd()}")

print(">>> Copying .env into local workspace...")
shutil.copy(ENV_PATH, ".env")

print(">>> Installing requirements...")
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
print(">>> Setup complete.")

>>> Loading secrets...
>>> Secrets loaded successfully.
>>> Configuring Git identity...
>>> Cloning repo...
>>> Working directory: /content/vlm-safety-reasoning
>>> Copying .env into local workspace...
>>> Installing requirements...
>>> Setup complete.


In [ ]:
import json
from pathlib import Path
from data.loader import load_processed_dataset
from models.model_loader import load_model_for_inference
from models.inference import run_inference_batched
from core.run_manifest import save_run_manifest
from core.config import load_config
from core.io import get_drive_path, ensure_dir

In [ ]:
MODEL_TIER = "2b"
N_SAMPLES = None # full test set
BATCH_SIZE = 80

DRIVE_RESULTS_DIR = get_drive_path("results", f"baseline_test_full_{MODEL_TIER}_second")
ensure_dir(DRIVE_RESULTS_DIR)
JSONL_OUTPUT_PATH = str(DRIVE_RESULTS_DIR / "predictions.jsonl")

In [ ]:
base_config = load_config(task="unified")

In [ ]:
run_config = {
    "experiment": "baseline_inference_full_testset_second_attemp",
    "model_tier": MODEL_TIER,
    "n_samples": N_SAMPLES,
    "batch_size": BATCH_SIZE,
    "max_new_tokens": base_config.get("max_new_tokens", 1000),
    "notes": "Full pass on test data.",
    "full_yaml_state": base_config
}
save_run_manifest(str(DRIVE_RESULTS_DIR), run_config)

2026-07-19 08:40:32 | INFO     | core.run_manifest:save_run_manifest:38 - Run manifest saved to /content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_2b_second/run_manifest.json


'/content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_2b_second/run_manifest.json'

In [ ]:
print("Loading dataset...")
splits = load_processed_dataset()
test_data = splits["test"]

Loading dataset...
2026-07-19 08:40:32 | INFO     | data.loader:load_processed_dataset:183 - Loading fully processed dataset from disk: /content/drive/MyDrive/vlm-finetuning-project1/datasets/processed
2026-07-19 08:40:57 | INFO     | data.loader:load_processed_dataset:187 - Loaded processed 'train' split: 6308 samples
2026-07-19 08:40:57 | INFO     | data.loader:load_processed_dataset:187 - Loaded processed 'val' split: 701 samples
2026-07-19 08:40:57 | INFO     | data.loader:load_processed_dataset:187 - Loaded processed 'test' split: 3004 samples


In [ ]:
img_first = test_data[0]["image"]
img_last = test_data[-1]["image"]

print(f"Smallest Image (Batch 1): {img_first.width}x{img_first.height} pixels")
print(f"Largest Image (Final Batch): {img_last.width}x{img_last.height} pixels")

Smallest Image (Batch 1): 320x240 pixels
Largest Image (Final Batch): 4416x3312 pixels


In [ ]:
model, tokenizer, info = load_model_for_inference(tier=MODEL_TIER)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
2026-07-19 08:41:27 | INFO     | models.model_loader:load_model_for_inference:186 - Loading model for inference: unsloth/Qwen3-VL-2B-Instruct with max_seq_length=4096
==((====))==  Unsloth 2026.7.3: Fast Qwen3_Vl patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

2026-07-19 08:41:48 | INFO     | models.model_loader:apply_pixel_bounds:232 - Applied image pixel bounds via size dict: min=200704, max=1204224


In [ ]:
run_config

{'experiment': 'baseline_inference_full_testset_second_attemp',
 'model_tier': '2b',
 'n_samples': None,
 'batch_size': 80,
 'max_new_tokens': 1000,
 'notes': 'Full pass on test data.',
 'full_yaml_state': {'drive_root': '/content/drive/MyDrive/vlm-finetuning-project1',
  'hf_org': 'nabeelshan',
  'wandb_project': 'vlm-safety-project1',
  'wandb_entity': 'nabeel-shan-research',
  'seed': 42,
  'dataset': {'hf_repo': 'LouisChen15/ConstructionSite',
   'raw_cache_subdir': 'datasets/raw',
   'cleaned_subdir': 'datasets/raw_cleaned',
   'processed_subdir': 'datasets/processed'},
  'paths': {'checkpoints_subdir': 'checkpoints',
   'results_subdir': 'results',
   'figures_subdir': 'figures',
   'logs_subdir': 'logs',
   'secrets_subdir': 'secrets'},
  'active_tier': '2b',
  'models': {'2b': {'hf_path': 'unsloth/Qwen3-VL-2B-Instruct',
    'short_name': 'qwen3vl-2b',
    'size_label': 'Small (2B)',
    'per_device_train_batch_size': 2,
    'gradient_accumulation_steps': 8},
   '4b': {'hf_path'

In [ ]:
# Inference
print(f"Starting batched inference. Results will stream to: {JSONL_OUTPUT_PATH}")
results = run_inference_batched(
    model=model,
    tokenizer=tokenizer,
    dataset=test_data,
    batch_size=run_config["batch_size"],
    max_new_tokens=run_config["max_new_tokens"],
    max_samples=run_config["n_samples"],
    output_path=JSONL_OUTPUT_PATH
)
print("Inference Complete!")

Starting batched inference. Results will stream to: /content/drive/MyDrive/vlm-finetuning-project1/results/baseline_test_full_2b_second/predictions.jsonl


Batched Inference: 100%|██████████| 38/38 [1:31:37<00:00, 144.68s/it]

2026-07-19 10:13:41 | INFO     | models.inference:run_inference_batched:337 - Batched inference complete: 3004 new samples processed.
Inference Complete!
